<a href="https://colab.research.google.com/github/chavattecedric-cloud/public/blob/main/Copie_de_%5BS%5DDojo_API_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Durée :**

- Matinée (  3 h)



**Problématique et objectifs :**

Comment récupérer et exploiter les données sur la qualité de l'air pour identifier les zones à risque en France ?

Pour cela, il faudra :

- Récupérer les indicateurs de qualité de l'air.
- Analyser les données pour identifier les zones géographiques à risque en France.


Dans ce Dojo, nous allons nous intéresser dans une première partie à l'extraction et le nettoyage de la donnée.

Dans le prochain Dojo, nous aurons à Explorer la donnée et indentifier les zones à risques en France

**Contexte**

Tout au long de ce challenge, nous allons utiliser l'API `World Air Quality - OpenAQ`, qui permet d'obtenir des informations sur la qualité de l'air. Vous pouvez commencer à l'explorer rapidement  

- [lien vers l'api](https://public.opendatasoft.com/explore/dataset/openaq/information/?disjunctive.measurements_parameter&disjunctive.location&disjunctive.city&sort=measurements_lastupdated)


Nous allons nous concentrer sur les données de la `France`, pour l'année `2025`

Les polluants à étudier sont :


- **O3 (Ozone)** : Gaz qui peut être nocif pour la santé lorsqu'il est présent en grande quantité au niveau du sol. L'ozone est un composant majeur du smog.
- **NO2 (Dioxyde d'azote)** : Gaz irritant pour les voies respiratoires, produit par la combustion des carburants fossiles.

- **PM10** : Particules de diamètre inférieur à 10 microns. Ces particules peuvent pénétrer les voies respiratoires et causer des problèmes de santé.
- **PM2.5** : Particules fines de diamètre inférieur à 2,5 microns. Elles sont particulièrement dangereuses car elles peuvent pénétrer profondément dans les poumons et même entrer dans la circulation sanguine.


# packages

In [1]:
# Pour faire des requêtes web (API)
import requests
# Pour la manipulation des dataframe
import pandas as pd
# Pour afficher joliment les json
from pprint import pprint
# Pour afficher des barres de progression dans une boucle
from tqdm import tqdm

# gestion des temps
import time

## Q1 : Récupération des données initiales

Votre première tâche est d'identifier l'URL de base de l'API permettant de récupérer les données de qualité de l'air, en vous aidant notamment de cette [page d'exploration OpenDataSoft](https://public.opendatasoft.com/explore/dataset/openaq/api/?sort=measurements_lastupdated).

Lorsque vous essaierez les différentes requêtes, vous verrez l'**URL mise à jour en bas de page**, ce qui vous aidera grandement.

La construction de l'url doit tenir compte des filtres suivants :

-   **country** : `FR` (France)
-   **date** : Ciblez l'année `2025`.
-   **polluants** : Les polluants cibles sont `['pm10', 'pm2.5', 'o3', 'no2']`.
-   **offset** : Définir à `0` pour obtenir la première "page" de résultats.
-   **limit** : Fixer le nombre maximum d'enregistrements par page à `50`.



---

Une fois que vous avez identifié l'URL de base et compris comment appliquer les filtres pour cette API , créez une fonction Python qui :


1.  Effectue une requête `GET` vers l'url définit de API en utilisant la bibliothèque `requests`.
2.  Vérifie si la requête a réussi (code statut 200).

3.  Retourne les données sous forme de JSON.

In [11]:
def api ():
  link = 'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/openaq/records?limit=50&offset=0&refine=country_name_en%3A%22France%22&refine=measurements_lastupdated%3A%222025%22&refine=measurements_parameter%3A%22PM10%22&refine=measurements_parameter%3A%22PM2.5%22&refine=measurements_parameter%3A%22O3%22&refine=measurements_parameter%3A%22NO2%22'
  r = requests.get(link)
  if r.status_code == 200:
    data=r.json()
  return data

In [18]:
api()

{'total_count': 1191,
 'results': [{'country': 'FR',
   'city': 'OUTREAU',
   'location': 'Outreau',
   'coordinates': {'lon': 1.5765700000004375, 'lat': 50.69434999996249},
   'measurements_parameter': 'O3',
   'measurements_sourcename': 'eea',
   'measurements_unit': 'µg/m³',
   'measurements_value': 67.0,
   'measurements_lastupdated': '2025-01-31T08:00:00+00:00',
   'country_name_en': 'France'},
  {'country': 'FR',
   'city': 'SAINT-LOUIS',
   'location': 'Sarda Garriga',
   'coordinates': {'lon': 55.40306500012004, 'lat': -21.277074999934275},
   'measurements_parameter': 'NO2',
   'measurements_sourcename': 'eea',
   'measurements_unit': 'µg/m³',
   'measurements_value': 8.6,
   'measurements_lastupdated': '2025-01-21T06:00:00+00:00',
   'country_name_en': 'France'},
  {'country': 'FR',
   'city': 'GRANDE-SYNTHE',
   'location': 'Grande-synthe',
   'coordinates': {'lon': 2.302149999732888, 'lat': 51.02446000023847},
   'measurements_parameter': 'NO2',
   'measurements_sourcename'

In [21]:
data=api()

## Q2 : Transformation en DataFrame

Votre prochaine tâche est de créer une fonction permettant de transformer le dictionnaire JSON obtenu de la requête API (que vous avez implémentée en Q1) en un DataFrame pandas.

In [22]:
def json_df (data):
  return pd.json_normalize(data,record_path='results')


In [23]:
json_df(data)

,country,city,location,measurements_parameter,measurements_sourcename,measurements_unit,measurements_value,measurements_lastupdated,country_name_en,coordinates.lon,coordinates.lat
0,FR,OUTREAU,Outreau,O3,eea,µg/m³,67.0,2025-01-31T08:00:00+00:00,France,1.576570,50.694350
1,FR,SAINT-LOUIS,Sarda Garriga,NO2,eea,µg/m³,8.6,2025-01-21T06:00:00+00:00,France,55.403065,-21.277075
2,FR,GRANDE-SYNTHE,Grande-synthe,NO2,eea,µg/m³,24.8,2025-01-31T08:00:00+00:00,France,2.302150,51.024460
3,FR,DOLE,Dole centre,O3,eea,µg/m³,56.8,2025-01-31T07:00:00+00:00,France,5.496389,47.096720
4,FR,SAINT-ESTÈVE,Perpignan St Esteve,O3,eea,µg/m³,55.3,2025-01-30T12:00:00+00:00,France,2.839880,42.719800
5,FR,NÎMES,Nimes Planas,PM10,eea,µg/m³,30.1,2025-01-27T08:00:00+00:00,France,4.361990,43.829900
6,FR,IFS,IFS Caen sud,O3,eea,µg/m³,51.5,2025-01-31T08:00:00+00:00,France,-0.352778,49.151670
7,FR,VESOUL,Vesoul Pres Caillet,PM2.5,eea,µg/m³,11.1,2025-01-30T12:00:00+00:00,France,6.157642,47.620020
8,FR,CAEN,Caen Chemin-Vert,NO2,eea,µg/m³,12.9,2025-01-31T08:00:00+00:00,France,-0.391111,49.192220
9,FR,CHAMPFORGEUIL,Champforgueil,PM2.5,eea,µg/m³,5.1,2025-01-30T12:00:00+00:00,France,4.835083,46.821056


## Q3 : Extraction complète

Maintenant que vous savez récupérer une seule page de données de l'API (Q1) et la transformer en DataFrame (Q2), l'étape suivante consiste à automatiser ce processus pour obtenir *toutes* les données disponibles sur toutes les pages.

Créez une nouvelle fonction qui :

1.  Utilise la fonction `get_content` de manière répétée pour récupérer les données page par page, en ajustant correctement le paramètre `offset`.
2. tilise la fonction `get_dataframe`  pour convertir le json en dataframe

3.  Assemble tous les dataframe  récupérés de chaque page en un seul dataframe


4.  Gère la condition d'arrêt pour savoir quand il n'y a plus de pages à récupérer (l'API OpenDataSoft indique généralement le nombre total d'enregistrements ).


In [24]:
def df_full():
  data_first=api() #stock le resultat de la requete api
  total_count=data_first['total_count'] #stock le nombre de ligne a obtenir pour nous servir a calculer le nombre de page plus tard
  limit=50
  nb_de_page= (total_count//limit)+1
  df_total=pd.DataFrame()
  for i in range (nb_de_page):#le numero de page est ajouter au lien api a chaque tour
    link = 'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/openaq/records?limit=50&offset='+str(i*50)+'&refine=country_name_en%3A%22France%22&refine=measurements_lastupdated%3A%222025%22&refine=measurements_parameter%3A%22PM10%22&refine=measurements_parameter%3A%22PM2.5%22&refine=measurements_parameter%3A%22O3%22&refine=measurements_parameter%3A%22NO2%22'
    r = requests.get(link) #verifie si acces ok
    if r.status_code == 200:
      data=r.json() #extrait les donnée
      df_page=json_df(data) # on sotcke la page récupéré
      df_total=pd.concat([df_total,df_page]) # on rassemble les page dans le dataframe final
  return df_total

In [25]:
df_full()

,country,city,location,measurements_parameter,measurements_sourcename,measurements_unit,measurements_value,measurements_lastupdated,country_name_en,coordinates.lon,coordinates.lat
0,FR,OUTREAU,Outreau,O3,eea,µg/m³,67.0,2025-01-31T08:00:00+00:00,France,1.576570,50.694350
1,FR,SAINT-LOUIS,Sarda Garriga,NO2,eea,µg/m³,8.6,2025-01-21T06:00:00+00:00,France,55.403065,-21.277075
2,FR,GRANDE-SYNTHE,Grande-synthe,NO2,eea,µg/m³,24.8,2025-01-31T08:00:00+00:00,France,2.302150,51.024460
3,FR,DOLE,Dole centre,O3,eea,µg/m³,56.8,2025-01-31T07:00:00+00:00,France,5.496389,47.096720
4,FR,SAINT-ESTÈVE,Perpignan St Esteve,O3,eea,µg/m³,55.3,2025-01-30T12:00:00+00:00,France,2.839880,42.719800
...,...,...,...,...,...,...,...,...,...,...,...
36,FR,QUIMPER,Quimper Zola,NO2,eea,µg/m³,28.0,2025-01-31T08:00:00+00:00,France,-4.097658,47.986607
37,FR,MONTCEAU-LES-MINES,Montceau-les-Mines 9me écluse,NO2,eea,µg/m³,11.8,2025-01-31T07:00:00+00:00,France,4.366893,46.678783
38,FR,COLMAR,Colmar Centre,NO2,eea,µg/m³,23.7,2025-01-31T08:00:00+00:00,France,7.350555,48.074444
39,FR,GENNEVILLIERS,GENNEVILLIERS,PM10,eea,µg/m³,7.2,2025-01-06T11:00:00+00:00,France,2.294350,48.930320


In [27]:
df_total=df_full()

In [30]:
df_total

,country,city,location,measurements_parameter,measurements_sourcename,measurements_unit,measurements_value,measurements_lastupdated,country_name_en,coordinates.lon,coordinates.lat
0,FR,OUTREAU,Outreau,O3,eea,µg/m³,67.0,2025-01-31T08:00:00+00:00,France,1.576570,50.694350
1,FR,SAINT-LOUIS,Sarda Garriga,NO2,eea,µg/m³,8.6,2025-01-21T06:00:00+00:00,France,55.403065,-21.277075
2,FR,GRANDE-SYNTHE,Grande-synthe,NO2,eea,µg/m³,24.8,2025-01-31T08:00:00+00:00,France,2.302150,51.024460
3,FR,DOLE,Dole centre,O3,eea,µg/m³,56.8,2025-01-31T07:00:00+00:00,France,5.496389,47.096720
4,FR,SAINT-ESTÈVE,Perpignan St Esteve,O3,eea,µg/m³,55.3,2025-01-30T12:00:00+00:00,France,2.839880,42.719800
...,...,...,...,...,...,...,...,...,...,...,...
36,FR,QUIMPER,Quimper Zola,NO2,eea,µg/m³,28.0,2025-01-31T08:00:00+00:00,France,-4.097658,47.986607
37,FR,MONTCEAU-LES-MINES,Montceau-les-Mines 9me écluse,NO2,eea,µg/m³,11.8,2025-01-31T07:00:00+00:00,France,4.366893,46.678783
38,FR,COLMAR,Colmar Centre,NO2,eea,µg/m³,23.7,2025-01-31T08:00:00+00:00,France,7.350555,48.074444
39,FR,GENNEVILLIERS,GENNEVILLIERS,PM10,eea,µg/m³,7.2,2025-01-06T11:00:00+00:00,France,2.294350,48.930320


## Q4 : Nettoyage du DataFrame

Une fois que vous avez récupéré toutes les données et les avez consolidées dans un DataFrame unique (grâce à Q3), il est temps de nettoyer ce DataFrame pour le rendre plus facile à utiliser pour l'analyse.

Créez une fonction Python qui prend le DataFrame complet en entrée et effectue les opérations suivantes :

-   **Supprimer les colonnes inutiles** : Éliminez les colonnes qui fournissent des informations redondantes pour l'analyse des zones à risque en France, comme :
    -   `location`
    -   `country`
    -   `country_name_en`
    -   `measurements_unit` (l'unité µg/m³ est fixe pour les polluants listés)

-   **Renommer des colonnes** : Donnez des noms plus clairs et uniformes aux colonnes importantes :
    -   `measurements_value` devient `measurements_value(µg/m³)`
    -   `coordinates.lon` devient `longitude`
    -   `coordinates.lat` devient `latitude`

-   **Convertir une colonne en datetime** : Assurez-vous que la colonne `measurements_lastupdated` est bien reconnue comme un type de date et heure par pandas.

-   **Extraire des composants date/heure** : À partir de la colonne `measurements_lastupdated` (une fois convertie), créez de nouvelles colonnes pour extraire des informations temporelles utiles :
    -   La date seule
    -   Le mois (numérique)
    -   L'heure (numérique)
    -   Le jour du mois (numérique)
    -   Le jour de la semaine (mumérique)

-   **Sauvegarder en CSV** : Enfin, sauvegardez ce DataFrame nettoyé dans un fichier CSV (par exemple, `air_quality_cleaned.csv`) pour une utilisation ultérieure, sans inclure l'index du DataFrame.




In [33]:
def cleanning(df):
  df=df.drop(columns=['location','country','country_name_en','measurements_unit'])
  df=df.rename(columns={'measurements_value':'measurements_value(µg/m³)','coordinates.lon':'longitude','coordinates.lat':'latitude'})
  df['measurements_lastupdated']=pd.to_datetime(df['measurements_lastupdated'])
  df['Date']=df['measurements_lastupdated'].dt.date
  df['Mois']=df['measurements_lastupdated'].dt.month
  df['Heure']=df['measurements_lastupdated'].dt.hour
  df['Jour']=df['measurements_lastupdated'].dt.day
  df['jour_semaine']=df['measurements_lastupdated'].dt.dayofweek
  df.to_csv("Dojo_API_Data_Cleaning.csv", index=False)
  return df

In [34]:
cleanning(df_total)

,city,measurements_parameter,measurements_sourcename,measurements_value(µg/m³),measurements_lastupdated,longitude,latitude,Date,Mois,Heure,Jour,jour_semaine
0,OUTREAU,O3,eea,67.0,2025-01-31 08:00:00+00:00,1.576570,50.694350,2025-01-31,1,8,31,4
1,SAINT-LOUIS,NO2,eea,8.6,2025-01-21 06:00:00+00:00,55.403065,-21.277075,2025-01-21,1,6,21,1
2,GRANDE-SYNTHE,NO2,eea,24.8,2025-01-31 08:00:00+00:00,2.302150,51.024460,2025-01-31,1,8,31,4
3,DOLE,O3,eea,56.8,2025-01-31 07:00:00+00:00,5.496389,47.096720,2025-01-31,1,7,31,4
4,SAINT-ESTÈVE,O3,eea,55.3,2025-01-30 12:00:00+00:00,2.839880,42.719800,2025-01-30,1,12,30,3
...,...,...,...,...,...,...,...,...,...,...,...,...
36,QUIMPER,NO2,eea,28.0,2025-01-31 08:00:00+00:00,-4.097658,47.986607,2025-01-31,1,8,31,4
37,MONTCEAU-LES-MINES,NO2,eea,11.8,2025-01-31 07:00:00+00:00,4.366893,46.678783,2025-01-31,1,7,31,4
38,COLMAR,NO2,eea,23.7,2025-01-31 08:00:00+00:00,7.350555,48.074444,2025-01-31,1,8,31,4
39,GENNEVILLIERS,PM10,eea,7.2,2025-01-06 11:00:00+00:00,2.294350,48.930320,2025-01-06,1,11,6,0
